# Phase 22 / Notebook 03 -- Phase 3 sandbox RL + final agent

Loads `checkpoint_phase2.pt`, runs REINFORCE / PPO-lite with a running EMA
baseline and a deterministic `search_ddg` tool sandbox, saves `final_model.pt`,
then runs `evaluation/final_agent_eval.py` with de-contamination checks.

Reward: `+1` correct final answer, `-0.5` per `[EDIT]` token, `+0.8` for a
verified 1-2 edit fix, `-2.0` for a wrong answer with no search, `+1.5` for
`Action: search_ddg("...")` **and** using its `Observation:` result.
`[PAST_ACTION: executed]` is immutable; only `[CURRENT_DRAFT: editable]` may
contain edits. This encodes **Search > Guess** and **Efficient Edit > Thrashing**.

In [ ]:
import json
import os
import pathlib
import subprocess
import sys

REPO = pathlib.Path(os.environ.get("AIRESEARCH_REPO", "/content/airesearch"))
if not (REPO / ".git").exists():
    subprocess.check_call(["git", "clone", "https://github.com/gabrielbosse1/airesearch.git", str(REPO)])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "datasets>=3,<5", "huggingface_hub>=0.25,<2", "pyyaml", "tqdm"])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("repo", REPO)

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("GPU", torch.cuda.get_device_name(DEVICE),
          "capability", torch.cuda.get_device_capability(DEVICE),
          "bf16_supported", torch.cuda.is_bf16_supported())
else:
    print("WARNING: no CUDA device. This notebook must run on a Colab T4 GPU for real training.")

## Load Phase 2 weights + verify the reward ordering

In [ ]:
from huggingface_hub import login

# Set these in Colab secrets / environment. The checkpoint repo is REQUIRED for
# robust resume when the Colab VM is recycled.
HF_REPO = os.environ.get("HF_REPO", "")
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)
cfg["checkpoints"]["hf_repo"] = HF_REPO or None
cfg["checkpoints"]["resume_from_hf"] = bool(HF_REPO)
print("HF checkpoint repo", cfg["checkpoints"]["hf_repo"] or "(not set - local resume only)")

In [ ]:
from training.phase22_common import hf_download, load_config

cfg = load_config(CONFIG_PATH)
cfg["checkpoints"]["hf_repo"] = HF_REPO or None
ck = cfg["checkpoints"]
phase2_path = pathlib.Path(ck["local_dir"]) / ck["phase2_name"]
print("fetch phase 2", hf_download(ck.get("hf_repo"),
                                   f"{ck['hf_path_in_repo']}/{ck['phase2_name']}",
                                   phase2_path))
assert phase2_path.is_file(), "Run notebook 02 first, or set HF_REPO to resume from the Hub."

In [ ]:
from training.phase22_reward import dominance_report

dom = dominance_report()
print(json.dumps({"ordering": dom["ordering"],
                  "all_satisfied": dom["all_satisfied"],
                  "weights": dom["weights"]}, indent=2))
assert dom["all_satisfied"], "Search>Guess / EfficientEdit>Thrashing must hold"

## Phase 3 RL

In [ ]:
from training.phase22_train import Phase22Trainer

MAX_STEPS = int(os.environ.get("PHASE3_MAX_STEPS", "0")) or None
trainer = Phase22Trainer(cfg, phase=3, device=DEVICE, data_mode="hub",
                         max_steps=MAX_STEPS, init_from_phase=2,
                         strict_phase_order=True)
report = trainer.run()
print(json.dumps({"steps": report["steps_completed"],
                  "initialisation": report["initialisation"],
                  "mean_reward": report.get("mean_reward"),
                  "dominance": report.get("dominance", {}).get("ordering"),
                  "checkpoint": report["checkpoint"]["path"]}, indent=2))

## Final model + de-contaminated evaluation

In [ ]:
path = trainer._checkpoint_path()
assert path.is_file(), f"missing {path}"
print("final_model.pt bytes", path.stat().st_size)
ck = cfg["checkpoints"]
if ck.get("hf_repo"):
    from training.phase22_common import hf_upload
    print("upload", hf_upload(path, ck["hf_repo"],
                              f"{ck['hf_path_in_repo']}/final_model.pt"))

# De-contaminated generalisation evaluation against the real model.
result = subprocess.run(
    [sys.executable, "-m", "evaluation.final_agent_eval",
     "--checkpoint", str(path), "--shard-dir", cfg["data"]["cache_dir"],
     "--root", ".", "--out", "runs/phase22/final_eval.json"],
    capture_output=True, text=True)
print(result.stdout)
print(result.stderr[-2000:])
eval_report = json.loads(pathlib.Path("runs/phase22/final_eval.json").read_text(encoding="utf8"))
print(json.dumps({"oracle_mode": eval_report["oracle_mode"],
                  "pass_rate": eval_report["pass_rate"],
                  "decontamination": eval_report["decontamination"]["status"],
                  "case_names": [r["name"] for r in eval_report["results"]]}, indent=2))

## Resume + honest reporting

Re-running loads `checkpoint_phase2.pt` first, then restores `final_model.pt`
state and skips completed RL steps. The evaluation report records
`oracle_mode` (scripted scorer proof) separately from real model generation,
and decontamination reports `unverified` unless Phase 22 training shards are
present to scan. Neither VRAM utilisation nor model quality is claimed in this
sandbox -- both must be measured here on the T4.